In [47]:
import pandas as pd
import re

# Charger le df concaténé des deux législatures
df = pd.read_csv(
    "../data/interim/extract_15_16_concat.csv",
    low_memory=False,
    dtype={
        "id_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
    },
)

print("Shape du df chargé : ", df.shape)

# NOTE : après test ne semble pas dramatique de ne pas prendre en compte
# seules 2 lignes changent
# les recodages "manuels" de PA repérés par ailleurs (voir autre notebook)
# ne changent rien non plus ici
# cf surtout des interruptions et ne change pas grand chose au regroup d'interventions
# quand erreur pas forcément de changement d'ID entre ou d'interv.

# # # par principe si on veut garder :
# # Stabiliser le id_orateur pour être au format AN
# df["id_orateur"] = "PA" + df["id_orateur"]
# # Remplacer les valeurs manquantes de id_acteur par id_orateur quand disponible
# df["id_acteur_originel"] = df["id_acteur"]  # garder une trace
# df["id_acteur"] = df["id_acteur"].combine_first(df["id_orateur"])

Shape du df chargé :  (1127838, 29)


# TODO :
Renvoyer aussi une variable de groupement pour eux :
```python
# (en plus de la variable originelle de la première ligne ?)
    "code_parole": lambda s: ", ".join(sorted(set(s.dropna().astype(str)))),
    "id_syceron": lambda s: s.dropna().unique().tolist(),
```

CG : Shape du df regroupé :  (968928, 34)


CG + ID : Shape du df regroupé :  (968930, 35)


CG + ID + NA : Shape du df regroupé :  (971491, 35)

In [ ]:
"""
==========================
Regroupe les lignes d'un CSV parlementaire pour fusionner les interventions
d'un même orateur interrompues par des INTERRUPTION_1_10.

Sortie : un CSV entrelacé avec :
  - une ligne par groupe d'intervention fusionnée (texte concaténé)
  - les lignes INTERRUPTION conservées telles quelles, intercalées dans l'ordre
"""

# ---------------------------------------------------------------------------
# Paramètres
# ---------------------------------------------------------------------------

# Codes considérés comme interruptions (conservés tels quels dans la sortie)
CODES_INTERRUPTION = {"INTERRUPTION_1_10"}

# Colonnes invariantes dans un groupe (on garde la valeur de la 1ère ligne)
COLS_META = [
    "uid",
    "SeanceRef",
    "SessionRef",
    "dateSeance",
    "dateSeanceJour",
    "numSeanceJour",
    "numSeance",
    "typeAssemblee",
    "legislature",
    "session",
    "nomFichierJo",
    "presidentSeance",
    "point_titre",
    "point_type",
    "valeur_ptsodj",
    "ordinal_prise",
    "ordre_absolu_seance",
    "id_acteur",
    "id_mandat",
    "code_grammaire",
    "code_style",
    "code_parole",
    "id_syceron",
    "roledebat",
    "nom_orateur",
    "qualite_orateur",
    "id_orateur",
    "stime",
]

# ---------------------------------------------------------------------------
# Fonctions principales
# ---------------------------------------------------------------------------


def regrouper(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prend un DataFrame trié par (uid, ordre_absolu_seance) et retourne
    un DataFrame entrelacé :
      - lignes d'intervention fusionnées (nb_fragments >= 1)
      - lignes d'interruption conservées telles quelles (nb_fragments = NaN)
    """
    cols_utiles = list(dict.fromkeys(COLS_META + ["texte"]))
    work = df[cols_utiles].copy()

    work["uid_norm"] = work["uid"].fillna("").astype(str)
    work["id_acteur_norm"] = work["id_acteur"].fillna("").astype(str)
    work["code_grammaire_norm"] = work["code_grammaire"].fillna("").astype(str)
    work["texte_norm"] = work["texte"].fillna("").astype(str)

    work = work.sort_values(["uid_norm", "ordre_absolu_seance"]).reset_index(drop=True)

    resultats = []  # liste finale (interventions + interruptions)
    groupe = None  # groupe en cours d'accumulation
    buffer_interruptions = []  # interruptions entre deux fragments du même orateur

    def ligne_sortie_depuis_base(base_row: dict) -> dict:
        r = base_row.copy()
        r["nb_fragments"] = pd.NA
        r["nb_interruptions_recues"] = pd.NA
        r["a_ete_interrompu"] = pd.NA
        r["codes_fragments"] = pd.NA
        r["changement_code_grammaire"] = pd.NA
        return r

    def clore_groupe(g: dict) -> dict:
        """
        Finalise un groupe. Les interruptions du buffer seront émises APRÈS dans le flux.
        """
        row = g["premiere_ligne"].copy()
        row["texte"] = " ".join(g["textes"])
        row["nb_fragments"] = g["nb_fragments"]
        row["nb_interruptions_recues"] = g["nb_interruptions_recues"]
        row["a_ete_interrompu"] = g["nb_interruptions_recues"] > 0
        row["codes_fragments"] = "|".join(g["codes"])
        row["changement_code_grammaire"] = len(set(g["codes"])) > 1
        return row

    records = work.to_dict("records")

    for row in records:
        cg = row["code_grammaire_norm"]
        acteur_str = row["id_acteur_norm"]
        uid_str = row["uid_norm"]

        # --- Cas 1 : interruption ---
        if cg in CODES_INTERRUPTION:
            if groupe is not None:
                # L'interruption est dans le contexte d'un groupe ouvert :
                # on l'ajoute au buffer (elle sera émise si le même orateur reprend)
                buffer_interruptions.append(row)
                groupe["nb_interruptions_recues"] += 1
            else:
                # Interruption hors contexte (cas rare) : on l'émet directement
                resultats.append(ligne_sortie_depuis_base(row))
            continue

        # --- Cas 2 : intervention principale ---
        if (
            groupe is not None
            and acteur_str != ""  # cf les nan convertis en ""
            and groupe["id_acteur"] == acteur_str
            and groupe["uid"] == uid_str
            and groupe["codes"][-1] == cg  # ajout condition code_grammaire constant
        ):
            # Même orateur, même séance, même code_grammaire -> on fusionne
            groupe["textes"].append(row["texte_norm"])
            groupe["codes"].append(cg)
            groupe["nb_fragments"] += 1
        else:
            # Nouvel orateur ou nouvelle séance ou changement de code_grammaire
            if groupe is not None:
                # Clore le groupe précédent
                resultats.append(clore_groupe(groupe))
                # Et les interruptions en buffer suivent le groupe
                for irr in buffer_interruptions:
                    resultats.append(ligne_sortie_depuis_base(irr))
                buffer_interruptions = []

            groupe = {
                "uid": uid_str,
                "id_acteur": acteur_str,
                "premiere_ligne": {col: row[col] for col in cols_utiles},
                "textes": [row["texte_norm"]],
                "codes": [cg],
                "nb_fragments": 1,
                "nb_interruptions_recues": 0,
            }
    # Clore le dernier groupe
    if groupe is not None:
        resultats.append(clore_groupe(groupe))
        for irr in buffer_interruptions:
            resultats.append(ligne_sortie_depuis_base(irr))

    return pd.DataFrame(resultats)


# ---------------------------------------------------------------------------
# Diagnostic : changements de code_grammaire pour un même acteur
# ---------------------------------------------------------------------------


def diagnostic_changements_code(df: pd.DataFrame) -> pd.DataFrame:
    """
    À lancer sur le df ORIGINAL (avant regroupement).
    Retourne les cas où un même acteur enchaîne deux code_grammaire différents
    sans interruption entre eux — utile pour investiguer les <interExtraction>.
    """
    main = df[~df["code_grammaire"].str.contains(PATTERN_INTERRUPTION, na=False)].copy()
    main = main.sort_values(["uid", "ordre_absolu_seance"])

    main["prev_acteur"] = main["id_acteur"].shift(1)
    main["prev_code"] = main["code_grammaire"].shift(1)
    main["prev_uid"] = main["uid"].shift(1)

    cas = main[
        (main["id_acteur"] == main["prev_acteur"])
        & (main["uid"] == main["prev_uid"])
        & (main["code_grammaire"] != main["prev_code"])
        & main["id_acteur"].notna()
        & (main["id_acteur"] != "")
    ][
        [
            "uid",
            "ordre_absolu_seance",
            "id_acteur",
            "nom_orateur",
            "prev_code",
            "code_grammaire",
            "texte",
        ]
    ].copy()

    cas.columns = [
        "uid",
        "ordre",
        "id_acteur",
        "nom_orateur",
        "code_precedent",
        "code_courant",
        "texte",
    ]
    return cas


In [50]:
df_group_bis = regrouper(df)
print("Shape du df regroupé : ", df_group_bis.shape)

Shape du df regroupé :  (971491, 38)


In [31]:
df_group_bis.to_csv("../data/interim/TEST_INTERRUPTIONS_BIS.csv", index=False)

In [ ]:
"""
regrouper_interventions.py
==========================
Regroupe les lignes d'un CSV parlementaire pour fusionner les interventions
d'un même orateur interrompues par des INTERRUPTION_*.

Sortie : un CSV entrelacé avec :
  - une ligne par groupe d'intervention fusionnée (texte concaténé)
  - les lignes INTERRUPTION conservées telles quelles, placées APRÈS leur groupe
"""

PATTERN_INTERRUPTION = "INTERRUPTION"


# ---------------------------------------------------------------------------
# Fonction principale (vectorisée)
# ---------------------------------------------------------------------------


def regrouper(df: pd.DataFrame) -> pd.DataFrame:
    """
    Regroupe les interventions interrompues de manière vectorisée.

    Stratégie :
      1. Séparer interventions principales et interruptions
      2. Sur les interventions, détecter les ruptures de groupe
         (changement d'acteur ou de séance) → groupe_id cumulatif
      3. Agréger par groupe_id (texte concaténé, first pour le reste)
      4. Propager le groupe_id aux interruptions via ffill
      5. Reconstruire le flux : intervention puis ses interruptions,
         triés par (uid, ordre_absolu_seance du 1er fragment, type)
    """
    df = df.sort_values(["uid", "ordre_absolu_seance"]).reset_index(drop=True)

    is_interrupt = df["code_grammaire"].str.contains(PATTERN_INTERRUPTION, na=False)
    main = df[~is_interrupt].copy()
    interrupts = df[is_interrupt].copy()

    # ------------------------------------------------------------------
    # 1. Calculer les groupe_id sur les interventions principales
    # ------------------------------------------------------------------
    acteur_norm = main["id_acteur"].fillna("").astype(str)
    uid_norm = main["uid"].fillna("").astype(str)
    code_norm = main["code_grammaire"].fillna("").astype(str)

    # Rupture si : changement d'acteur, de séance, de code_grammaire,
    # ou acteur vide (lignes sans id_acteur ne sont jamais fusionnées)
    rupture = (
        (acteur_norm != acteur_norm.shift(1))
        | (uid_norm != uid_norm.shift(1))
        | (code_norm != code_norm.shift(1))
        | (acteur_norm == "")
    )
    main["groupe_id"] = rupture.cumsum()

    # ------------------------------------------------------------------
    # 2. Agréger les interventions par groupe
    # ------------------------------------------------------------------
    agg = {
        c: "first"
        for c in main.columns
        if c not in ["texte", "code_grammaire", "groupe_id"]
    }
    agg["texte"] = lambda s: " ".join(s.dropna().astype(str))
    agg["code_grammaire"] = lambda s: "|".join(s.dropna().astype(str))

    grouped = main.groupby("groupe_id", sort=False).agg(agg).reset_index(drop=True)

    # nb_fragments
    grouped["nb_fragments"] = main.groupby("groupe_id", sort=False).size().values

    # codes_fragments / changement_code_grammaire / code_grammaire (1er fragment)
    grouped.rename(columns={"code_grammaire": "codes_fragments"}, inplace=True)
    grouped["code_grammaire"] = grouped["codes_fragments"].str.split("|").str[0]
    grouped["changement_code_grammaire"] = grouped["codes_fragments"].apply(
        lambda s: len(set(s.split("|"))) > 1
    )

    # ------------------------------------------------------------------
    # 3. Propager groupe_id aux interruptions via ffill
    # ------------------------------------------------------------------
    df["groupe_id"] = pd.NA
    df.loc[~is_interrupt, "groupe_id"] = main["groupe_id"].values
    df["groupe_id"] = df["groupe_id"].ffill()

    # Compter les interruptions par groupe
    interrupt_counts = (
        df[is_interrupt]
        .groupby("groupe_id", sort=False)
        .size()
        .rename("nb_interruptions_recues")
        .reset_index()
    )

    # Rattacher le groupe_id au grouped pour le merge
    groupe_ids = main.groupby("groupe_id", sort=False)["groupe_id"].first().values
    grouped["groupe_id"] = groupe_ids
    grouped = grouped.merge(interrupt_counts, on="groupe_id", how="left")
    grouped["nb_interruptions_recues"] = (
        grouped["nb_interruptions_recues"].fillna(0).astype(int)
    )
    grouped["a_ete_interrompu"] = grouped["nb_interruptions_recues"] > 0
    grouped.drop(columns="groupe_id", inplace=True)

    # ------------------------------------------------------------------
    # 4. Reconstruire le flux ordonné
    # ------------------------------------------------------------------
    # tie-breaker : 0 = intervention, 1 = interruption
    # → intervention toujours avant ses interruptions à même ordre
    for col in [
        "nb_fragments",
        "nb_interruptions_recues",
        "a_ete_interrompu",
        "codes_fragments",
        "changement_code_grammaire",
    ]:
        interrupts[col] = pd.NA

    grouped["_sort_ordre"] = grouped["ordre_absolu_seance"]
    grouped["_sort_type"] = 0

    interrupts["_sort_ordre"] = interrupts["ordre_absolu_seance"]
    interrupts["_sort_type"] = 1

    result = (
        pd.concat([grouped, interrupts], ignore_index=True)
        .sort_values(["uid", "_sort_ordre", "_sort_type"])
        .drop(columns=["_sort_ordre", "_sort_type"])
        .reset_index(drop=True)
    )

    return result


# ---------------------------------------------------------------------------
# Diagnostic : changements de code_grammaire pour un même acteur
# ---------------------------------------------------------------------------


def diagnostic_changements_code(df: pd.DataFrame) -> pd.DataFrame:
    """
    À lancer sur le df ORIGINAL (avant regroupement).
    Retourne les cas où un même acteur enchaîne deux code_grammaire différents
    sans interruption entre eux — utile pour investiguer les <interExtraction>.
    """
    main = df[~df["code_grammaire"].str.contains(PATTERN_INTERRUPTION, na=False)].copy()
    main = main.sort_values(["uid", "ordre_absolu_seance"])

    main["prev_acteur"] = main["id_acteur"].shift(1)
    main["prev_code"] = main["code_grammaire"].shift(1)
    main["prev_uid"] = main["uid"].shift(1)

    cas = main[
        (main["id_acteur"] == main["prev_acteur"])
        & (main["uid"] == main["prev_uid"])
        & (main["code_grammaire"] != main["prev_code"])
        & main["id_acteur"].notna()
        & (main["id_acteur"] != "")
    ][
        [
            "uid",
            "ordre_absolu_seance",
            "id_acteur",
            "nom_orateur",
            "prev_code",
            "code_grammaire",
            "texte",
        ]
    ].copy()

    cas.columns = [
        "uid",
        "ordre",
        "id_acteur",
        "nom_orateur",
        "code_precedent",
        "code_courant",
        "texte",
    ]
    return cas


In [33]:
df_group_ter = regrouper(df)
print("Shape du df regroupé : ", df_group_ter.shape)

/var/folders/rq/xsj46x_s2rg87wdksm1_jl3c0000gn/T/ipykernel_4992/2392771363.py:79: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['groupe_id'] = df['groupe_id'].ffill()


Shape du df regroupé :  (971491, 35)


In [34]:
df_group_ter.to_csv("../data/interim/TEST_INTERRUPTIONS_TER.csv", index=False)